# Text to Graph using LLMGraphTransformer
Convert text to knowledge graph using Google Gemini and store in Neo4j

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph
from langchain_core.documents import Document

load_dotenv()

True

In [2]:
# Initialize Neo4j connection
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE")
)

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [3]:
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

# Initialize LLMGraphTransformer
transformer = LLMGraphTransformer(llm=llm)

In [4]:
# Function to convert text to graph and store in Neo4j
def text_to_graph(text):
    documents = [Document(page_content=text)]
    graph_documents = transformer.convert_to_graph_documents(documents)
    graph.add_graph_documents(graph_documents)
    print(f"Added {len(graph_documents)} graph documents to Neo4j")
    return graph_documents

In [5]:
# Example: Convert sample text to graph
sample_text = """
Marie Curie was a Polish physicist and chemist who conducted pioneering research on radioactivity.
She was the first woman to win a Nobel Prize and the only person to win Nobel Prizes in two scientific fields.
Marie Curie worked at the University of Paris and discovered the elements polonium and radium.
"""

graph_docs = text_to_graph(sample_text)

Added 1 graph documents to Neo4j


In [6]:
# Function to query Neo4j database
def query_graph(cypher_query):
    result = graph.query(cypher_query)
    return result

In [7]:
# Example queries

# Get all nodes
print("All nodes:")
print(query_graph("MATCH (n) RETURN n LIMIT 10"))

# Get all relationships
print("\nAll relationships:")
print(query_graph("MATCH (a)-[r]->(b) RETURN a, r, b LIMIT 10"))

All nodes:
[{'n': {'id': 'Marie Curie'}}, {'n': {'id': 'Poland'}}, {'n': {'id': 'Radioactivity'}}, {'n': {'id': 'Nobel Prize'}}, {'n': {'id': 'University Of Paris'}}, {'n': {'id': 'Polonium'}}, {'n': {'id': 'Radium'}}, {'n': {'id': 'Physics'}}, {'n': {'id': 'Chemistry'}}]

All relationships:
[{'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'NATIONALITY', {'id': 'Poland'}), 'b': {'id': 'Poland'}}, {'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'RESEARCH', {'id': 'Radioactivity'}), 'b': {'id': 'Radioactivity'}}, {'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'RECIPIENT', {'id': 'Nobel Prize'}), 'b': {'id': 'Nobel Prize'}}, {'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'AFFILIATION', {'id': 'University Of Paris'}), 'b': {'id': 'University Of Paris'}}, {'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'DISCOVERY', {'id': 'Polonium'}), 'b': {'id': 'Polonium'}}, {'a': {'id': 'Marie Curie'}, 'r': ({'id': 'Marie Curie'}, 'DISCOVERY', {'id': '

In [8]:
# Custom query example
custom_query = """
MATCH (n)
RETURN labels(n) as NodeType, count(*) as Count
"""
print("Node type counts:")
print(query_graph(custom_query))

Node type counts:
[{'NodeType': ['Person'], 'Count': 1}, {'NodeType': ['Location'], 'Count': 1}, {'NodeType': ['Concept'], 'Count': 1}, {'NodeType': ['Award'], 'Count': 1}, {'NodeType': ['Organization'], 'Count': 1}, {'NodeType': ['Element'], 'Count': 2}, {'NodeType': ['Field'], 'Count': 2}]
